# LAB SLOT 18 - SINH CÂU TIẾNG VIỆT CÓ DẤU (~40 TỪ)
**Sinh viên:** Nguyễn Văn Anh Duy  
**MSSV:** SE181823  
**Lớp:** AI1803

---
**Dataset:** ViDiacritics_test.csv (cột with_diacritics)

## Mục tiêu bài lab
- Xây dựng hệ thống sinh câu tiếng Việt có dấu từ dữ liệu thực tế.
- Hỗ trợ 2 chế độ đầu vào:
  - Chế độ A: Nhập 1 từ.
  - Chế độ B: Nhập 1 câu bất kỳ.
- Mỗi lần sinh trả về 1 câu có độ dài mục tiêu khoảng 40 từ (ưu tiên 38-42 từ).

## Yêu cầu cần đạt
### Yêu cầu 1: Tiền xử lý dữ liệu tiếng Việt
- Chuẩn hóa văn bản về chữ thường.
- Giữ dấu tiếng Việt, chuẩn hóa khoảng trắng.
- Tokenize theo từ và lọc nhiễu cơ bản.

### Yêu cầu 2: Xây dựng mô hình liên kết từ
- Xây dựng Bigram Graph từ dữ liệu đã làm sạch.
- Mở rộng thêm Trigram để tăng độ mượt ngữ cảnh.

### Yêu cầu 3: Thuật toán sinh câu
- Sinh câu từ 1 từ hoặc 1 câu đầu vào.
- Có cơ chế chống lặp vô hạn (visited set).
- Có cơ chế fallback khi gặp dead-end.
- Có điều khiển coherence theo chủ đề để giảm nhảy ngữ cảnh.

### Yêu cầu 4: Đánh giá đầu ra
- Chạy tối thiểu 10 lần cho mỗi chế độ.
- Thống kê số từ và tỷ lệ đạt ngưỡng 38-42 từ.

### Yêu cầu 5: Báo cáo nộp bài
- Trình bày chức năng theo từng yêu cầu.
- Có minh họa output cho cả 2 chế độ đầu vào.
- Giải thích nguyên lý hoạt động và cơ chế tối ưu chất lượng.

## 1. Import Libraries & Cấu hình

Cell này khai báo thư viện, seed ngẫu nhiên và các tham số mục tiêu độ dài câu.

In [25]:
import re
import random
from pathlib import Path
from collections import defaultdict, Counter

import pandas as pd

RANDOM_SEED = 42
TARGET_WORDS = 40
MIN_WORDS = 38
MAX_WORDS = 42
MAX_ROWS = None  # Đặt số nguyên (ví dụ 100000) nếu muốn giới hạn để chạy nhanh hơn

random.seed(RANDOM_SEED)
pd.set_option("display.max_colwidth", 200)

## 2. Yêu cầu 1 - Tiền xử lý dữ liệu tiếng Việt

**Mục tiêu:** Làm sạch dữ liệu để giảm token nhiễu và tăng tính tự nhiên cho câu sinh.

**Các bước chính:**
1. Chuẩn hóa văn bản (lowercase, khoảng trắng, loại URL/HTML).
2. Tokenize tiếng Việt theo regex giữ ký tự có dấu.
3. Lọc từ theo tần suất, độ dài và đặc trưng tiếng Việt.
4. Trả về corpus đã làm sạch để xây mô hình ngôn ngữ.

In [ ]:
DATA_PATH = Path("data/ViDiacritics_test.csv")

# Các ngưỡng lọc giúp bớt token nhiễu và tăng tính tự nhiên của câu sinh
MIN_TOKEN_FREQ = 3
MIN_TOKEN_LEN = 2
MAX_TOKEN_LEN = 20
MAX_NON_VI_RATIO = 0.35  # Nếu câu có quá nhiều token không giống tiếng Việt thì loại

NOISE_TOKENS = {
    "http", "https", "www", "com", "vn", "net", "org", "html", "amp", "img", "jpg", "png"
}

VI_VOWEL_PATTERN = re.compile(
    r"[aeiouyăâđêôơưáàảãạắằẳẵặấầẩẫậéèẻẽẹếềểễệ"
    r"íìỉĩịóòỏõọốồổỗộớờởỡợúùủũụứừửữựýỳỷỹỵ]"
)


def clean_vietnamese_text(text: str) -> str:
    text = str(text).strip().lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[_\-–—]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def split_sentences(text: str):
    parts = re.split(r"[.!?;:\n\r]+", text)
    return [p.strip() for p in parts if p.strip()]


def tokenize_vietnamese(sentence: str):
    # Giữ chữ cái tiếng Việt có dấu và chữ đ
    return re.findall(r"[a-zà-ỹđ]+", sentence.lower())


def is_good_token(token: str, token_freq: Counter) -> bool:
    if token in NOISE_TOKENS:
        return False
    if not (MIN_TOKEN_LEN <= len(token) <= MAX_TOKEN_LEN):
        return False
    if token_freq[token] < MIN_TOKEN_FREQ:
        return False
    if not VI_VOWEL_PATTERN.search(token):
        return False
    return True


def load_and_preprocess(csv_path: Path, max_rows=None):
    if max_rows is None:
        df = pd.read_csv(csv_path)
    else:
        df = pd.read_csv(csv_path, nrows=max_rows)

    if "with_diacritics" in df.columns:
        text_col = "with_diacritics"
    else:
        object_cols = [c for c in df.columns if df[c].dtype == "object"]
        if not object_cols:
            raise ValueError("Không tìm thấy cột văn bản trong dataset.")
        text_col = object_cols[0]

    raw_sentences = []
    raw_counter = Counter()

    for raw_text in df[text_col].dropna().astype(str):
        cleaned = clean_vietnamese_text(raw_text)
        for sent in split_sentences(cleaned):
            tokens = tokenize_vietnamese(sent)
            if len(tokens) >= 2:
                raw_sentences.append(tokens)
                raw_counter.update(tokens)

    tokenized_sentences = []
    filtered_counter = Counter()

    for sent in raw_sentences:
        filtered = [tk for tk in sent if is_good_token(tk, raw_counter)]
        if len(filtered) < 2:
            continue

        non_vi_like = sum(1 for tk in filtered if not VI_VOWEL_PATTERN.search(tk))
        if len(filtered) > 0 and (non_vi_like / len(filtered)) > MAX_NON_VI_RATIO:
            continue

        tokenized_sentences.append(filtered)
        filtered_counter.update(filtered)

    return tokenized_sentences, filtered_counter, text_col, len(raw_sentences), len(raw_counter)


tokenized_sentences, vocab_counter, text_col, raw_sentence_count, raw_vocab_count = load_and_preprocess(DATA_PATH, MAX_ROWS)

print(f"Đã đọc cột văn bản: {text_col}")
print(f"Số câu trước lọc: {raw_sentence_count:,}")
print(f"Số câu sau lọc chất lượng: {len(tokenized_sentences):,}")
print(f"Số từ vựng trước lọc: {raw_vocab_count:,}")
print(f"Số từ vựng sau lọc: {len(vocab_counter):,}")
print(f"10 từ phổ biến nhất: {vocab_counter.most_common(10)}")

## 3. Yêu cầu 2 & 3 - Xây dựng mô hình và thuật toán sinh câu

**Thành phần mô hình:**
- Bigram Graph: học quan hệ từ kế tiếp.
- Reverse Bigram: hỗ trợ gom ngữ cảnh theo chủ đề.
- Trigram Graph: tăng độ mượt khi sinh chuỗi.

**Chiến lược sinh câu:**
1. Nhận seed từ đầu vào (1 từ hoặc 1 câu).
2. Tạo topic context để giữ coherence theo chủ đề.
3. Sinh từng từ bằng trọng số xác suất + chất lượng token.
4. Chống lặp bằng visited set.
5. Dùng fallback khi dead-end để vẫn đảm bảo độ dài mục tiêu.

In [21]:
CONNECTOR_TOKENS = ["và", "nhưng", "nên", "vì", "khi", "để", "trong", "với", "theo", "tại"]
TOP_K_NEXT = 8
TOPIC_MAX_TERMS = 120
OFF_TOPIC_PENALTY = 0.25
COHERENCE_INTERVAL = 8

FUNCTION_WORDS = {
    "và", "nhưng", "nên", "vì", "khi", "để", "trong", "với", "theo", "tại", "của", "cho",
    "là", "đã", "đang", "sẽ", "có", "không", "một", "những", "các", "trên", "dưới", "về"
}


def build_language_graphs(tokenized_data):
    bigram_graph = defaultdict(Counter)
    reverse_bigram_graph = defaultdict(Counter)
    trigram_graph = defaultdict(Counter)
    start_counter = Counter()

    for sent in tokenized_data:
        if sent:
            start_counter[sent[0]] += 1
        for w1, w2 in zip(sent, sent[1:]):
            bigram_graph[w1][w2] += 1
            reverse_bigram_graph[w2][w1] += 1
        for w1, w2, w3 in zip(sent, sent[1:], sent[2:]):
            trigram_graph[(w1, w2)][w3] += 1

    return bigram_graph, reverse_bigram_graph, trigram_graph, start_counter


def _token_quality(token: str, token_freq: Counter) -> float:
    freq = token_freq.get(token, 1)
    return 1.0 + min(2.0, (freq ** 0.3) / 3.0)


def build_topic_context(seed_tokens, bigram_graph, reverse_bigram_graph, trigram_graph, token_freq):
    topic_scores = Counter()

    valid_seed = [tk for tk in seed_tokens if tk in bigram_graph]
    for tk in valid_seed:
        topic_scores[tk] += 6.0

        for nxt, c in bigram_graph[tk].most_common(10):
            topic_scores[nxt] += min(3.0, c ** 0.25)

        for prev, c in reverse_bigram_graph[tk].most_common(8):
            topic_scores[prev] += min(2.5, c ** 0.25)

    for w1, w2 in zip(valid_seed, valid_seed[1:]):
        tri_counter = trigram_graph.get((w1, w2), Counter())
        for nxt, c in tri_counter.most_common(6):
            topic_scores[nxt] += min(3.2, c ** 0.25)

    if not topic_scores:
        return None

    # Ưu tiên token phổ biến vừa phải để tăng độ tự nhiên
    rescored = Counter()
    for tk, sc in topic_scores.items():
        rescored[tk] = sc * _token_quality(tk, token_freq)

    top_topic = [w for w, _ in rescored.most_common(TOPIC_MAX_TERMS)]
    topic_words = set(top_topic)
    return {"topic_words": topic_words, "topic_scores": rescored}


def topic_multiplier(token: str, topic_context):
    if topic_context is None:
        return 1.0

    topic_words = topic_context["topic_words"]
    topic_scores = topic_context["topic_scores"]

    if token in topic_words:
        return 1.2 + min(1.8, topic_scores[token] / 10.0)
    if token in FUNCTION_WORDS:
        return 1.0
    return OFF_TOPIC_PENALTY


def _sample_from_counter(counter_obj: Counter, visited: set, token_freq: Counter, topic_context=None, top_k=TOP_K_NEXT):
    candidates = [(w, c) for w, c in counter_obj.items() if w not in visited]
    if not candidates:
        return None

    candidates.sort(key=lambda x: x[1], reverse=True)
    candidates = candidates[:top_k]

    words = [w for w, _ in candidates]
    weights = []
    for w, c in candidates:
        wgt = c * _token_quality(w, token_freq) * topic_multiplier(w, topic_context)
        weights.append(max(1e-8, wgt))

    return random.choices(words, weights=weights, k=1)[0]


def choose_next_word(prev_word, current_word, visited, bigram_graph, trigram_graph, token_freq, topic_context=None):
    if prev_word is not None:
        tri_counter = trigram_graph.get((prev_word, current_word), Counter())
        nxt = _sample_from_counter(tri_counter, visited, token_freq, topic_context)
        if nxt is not None:
            return nxt

    bi_counter = bigram_graph.get(current_word, Counter())
    return _sample_from_counter(bi_counter, visited, token_freq, topic_context)


def pick_topic_anchor(visited: set, topic_context, bigram_graph):
    if topic_context is None:
        return None

    topic_scores = topic_context["topic_scores"]
    candidates = [w for w, _ in topic_scores.most_common(40) if w not in visited and w in bigram_graph and len(bigram_graph[w]) > 0]
    if not candidates:
        return None
    return random.choice(candidates[:10])


def fallback_anchor(visited: set, bigram_graph, start_counter, topic_context=None):
    topic_anchor = pick_topic_anchor(visited, topic_context, bigram_graph)
    if topic_anchor is not None:
        return topic_anchor

    connector_candidates = [w for w in CONNECTOR_TOKENS if w in bigram_graph and w not in visited]
    if connector_candidates:
        return random.choice(connector_candidates)

    start_candidates = [(w, c) for w, c in start_counter.items() if w not in visited]
    if start_candidates:
        words = [w for w, _ in start_candidates]
        weights = [c for _, c in start_candidates]
        return random.choices(words, weights=weights, k=1)[0]

    candidates = [w for w in bigram_graph.keys() if w not in visited and len(bigram_graph[w]) > 0]
    if not candidates:
        return None
    return random.choice(candidates)


def finalize_sentence(tokens):
    if not tokens:
        return ""
    s = " ".join(tokens).strip()
    if not s:
        return ""
    s = s[0].upper() + s[1:]
    if s[-1] not in ".!?":
        s += "."
    return s


def extend_phrase_to_target(
    phrase,
    bigram_graph,
    trigram_graph,
    start_counter,
    token_freq,
    topic_context=None,
    target_words=TARGET_WORDS,
    min_words=MIN_WORDS,
    max_words=MAX_WORDS,
):
    visited = set(phrase)
    dead_end_count = 0
    max_steps = target_words * 5

    prev_word = phrase[-2] if len(phrase) >= 2 else None
    current = phrase[-1]

    while len(phrase) < target_words and max_steps > 0:
        max_steps -= 1

        # Định kỳ kéo câu quay lại chủ đề để giảm nhảy ngữ cảnh
        if topic_context is not None and len(phrase) % COHERENCE_INTERVAL == 0:
            anchor = pick_topic_anchor(visited, topic_context, bigram_graph)
            if anchor is not None:
                phrase.append(anchor)
                visited.add(anchor)
                prev_word, current = current, anchor
                continue

        nxt = choose_next_word(
            prev_word, current, visited,
            bigram_graph, trigram_graph,
            token_freq, topic_context
        )
        if nxt is None:
            dead_end_count += 1
            anchor = fallback_anchor(visited, bigram_graph, start_counter, topic_context)
            if anchor is None:
                break
            phrase.append(anchor)
            visited.add(anchor)
            prev_word, current = current, anchor
            continue

        phrase.append(nxt)
        visited.add(nxt)
        prev_word, current = current, nxt

    if len(phrase) > max_words:
        phrase = phrase[:max_words]

    while len(phrase) < min_words:
        anchor = fallback_anchor(set(phrase), bigram_graph, start_counter, topic_context)
        if anchor is None:
            break
        phrase.append(anchor)

    return phrase, dead_end_count


def normalize_seed_word(seed_word: str, bigram_graph, start_counter):
    tokens = tokenize_vietnamese(seed_word)
    for tk in tokens:
        if tk in bigram_graph:
            return tk

    if start_counter:
        words = list(start_counter.keys())
        weights = list(start_counter.values())
        return random.choices(words, weights=weights, k=1)[0]

    return random.choice(list(bigram_graph.keys()))


def generate_from_word(seed_word: str, bigram_graph, reverse_bigram_graph, trigram_graph, start_counter, token_freq):
    start = normalize_seed_word(seed_word, bigram_graph, start_counter)
    phrase = [start]

    topic_context = build_topic_context([start], bigram_graph, reverse_bigram_graph, trigram_graph, token_freq)

    phrase, dead_ends = extend_phrase_to_target(
        phrase=phrase,
        bigram_graph=bigram_graph,
        trigram_graph=trigram_graph,
        start_counter=start_counter,
        token_freq=token_freq,
        topic_context=topic_context,
    )

    return {
        "mode": "word",
        "input": seed_word,
        "start": start,
        "dead_ends": dead_ends,
        "tokens": phrase,
        "sentence": finalize_sentence(phrase),
        "word_count": len(phrase),
    }


def generate_from_sentence(seed_sentence: str, bigram_graph, reverse_bigram_graph, trigram_graph, start_counter, token_freq):
    raw_tokens = tokenize_vietnamese(clean_vietnamese_text(seed_sentence))

    prefix = []
    seen = set()
    for tk in raw_tokens:
        if tk in bigram_graph and tk not in seen:
            prefix.append(tk)
            seen.add(tk)

    if not prefix:
        return generate_from_word("", bigram_graph, reverse_bigram_graph, trigram_graph, start_counter, token_freq)

    if len(prefix) > MAX_WORDS:
        prefix = prefix[:MAX_WORDS]

    topic_context = build_topic_context(prefix, bigram_graph, reverse_bigram_graph, trigram_graph, token_freq)

    phrase, dead_ends = extend_phrase_to_target(
        phrase=prefix.copy(),
        bigram_graph=bigram_graph,
        trigram_graph=trigram_graph,
        start_counter=start_counter,
        token_freq=token_freq,
        topic_context=topic_context,
    )

    return {
        "mode": "sentence",
        "input": seed_sentence,
        "start": phrase[0],
        "dead_ends": dead_ends,
        "tokens": phrase,
        "sentence": finalize_sentence(phrase),
        "word_count": len(phrase),
    }


# Build mô hình ngôn ngữ
bigram_graph, reverse_bigram_graph, trigram_graph, start_counter = build_language_graphs(tokenized_sentences)
graph = bigram_graph
all_nodes = list(bigram_graph.keys())

edge_count = sum(len(nexts) for nexts in bigram_graph.values())
tri_edge_count = sum(len(nexts) for nexts in trigram_graph.values())

print(f"Số node trong bigram graph: {len(all_nodes):,}")
print(f"Số cạnh bigram: {edge_count:,}")
print(f"Số trạng thái trigram: {len(trigram_graph):,}")
print(f"Số cạnh trigram: {tri_edge_count:,}")

Số node trong bigram graph: 25,198
Số cạnh bigram: 1,638,854
Số trạng thái trigram: 1,571,663
Số cạnh trigram: 6,152,966


## 4. Demo 2 chế độ đầu vào

**Chế độ A - Word Seed**
- Người dùng nhập 1 từ tiếng Việt có dấu.
- Hệ thống kiểm tra hợp lệ trước khi sinh.

**Chế độ B - Sentence Seed**
- Người dùng nhập 1 câu tiếng Việt bất kỳ.
- Hệ thống trích ngữ cảnh đầu vào và sinh tiếp tới độ dài mục tiêu.

In [22]:
# Cell nhập tay cho 2 chế độ.
# Nếu vừa mở kernel mới, hãy chạy Cell 3 -> Cell 7 trước.

VI_DIACRITIC_PATTERN = re.compile(r"[àáảãạăắằẳẵặâấầẩẫậđèéẻẽẹêếềểễệìíỉĩịòóỏõọôốồổỗộơớờởỡợùúủũụưứừửữựỳýỷỹỵ]")


def is_vietnamese_diacritic_word(word: str) -> bool:
    return bool(word) and bool(VI_DIACRITIC_PATTERN.search(word.lower())) and (word.lower() in bigram_graph)


def ask_seed_word_with_diacritics(default_word: str = "học") -> str:
    while True:
        user_input = input(
            f"Nhập 1 từ tiếng Việt có dấu (chế độ A) [mặc định: {default_word}]: "
        ).strip().lower()

        if not user_input:
            user_input = default_word

        if is_vietnamese_diacritic_word(user_input):
            return user_input

        print("Từ không hợp lệ. Vui lòng nhập từ tiếng Việt có dấu và có trong dữ liệu.")


sample_word_input = ask_seed_word_with_diacritics(default_word="học")

sample_sentence_input = input(
    "Nhập 1 câu tiếng Việt (chế độ B) [mặc định: hôm nay tôi muốn học xử lý ngôn ngữ tự nhiên]: "
).strip()
if not sample_sentence_input:
    sample_sentence_input = "hôm nay tôi muốn học xử lý ngôn ngữ tự nhiên"

result_word = generate_from_word(
    sample_word_input,
    bigram_graph,
    reverse_bigram_graph,
    trigram_graph,
    start_counter,
    vocab_counter,
)

print("=== CHẾ ĐỘ A: NHẬP 1 TỪ ===")
print("Input:", result_word["input"])
print("Start token:", result_word["start"])
print("Dead-end count:", result_word["dead_ends"])
print("Word count:", result_word["word_count"])
print("Output:", result_word["sentence"])
print()

result_sentence = generate_from_sentence(
    sample_sentence_input,
    bigram_graph,
    reverse_bigram_graph,
    trigram_graph,
    start_counter,
    vocab_counter,
)

print("=== CHẾ ĐỘ B: NHẬP 1 CÂU ===")
print("Input:", result_sentence["input"])
print("Start token:", result_sentence["start"])
print("Dead-end count:", result_sentence["dead_ends"])
print("Word count:", result_sentence["word_count"])
print("Output:", result_sentence["sentence"])

Từ không hợp lệ. Vui lòng nhập từ tiếng Việt có dấu và có trong dữ liệu.
=== CHẾ ĐỘ A: NHẬP 1 TỪ ===
Input: đầu
Start token: đầu
Dead-end count: 0
Word count: 40
Output: Đầu tiên của năm tại tỉnh hà giang từ sau khi bị bắt cóc tống tiền hàng tháng và có một người đàn ông tuần đẻ mỗi ngày để làm đẹp với dẫn đến sự phát triển kinh tế thế.

=== CHẾ ĐỘ B: NHẬP 1 CÂU ===
Input: trí tuệ nhân tạo
Start token: trí
Dead-end count: 0
Word count: 40
Output: Trí tuệ nhân tạo có thể bị phạt trong đó điểm chuẩn năm của mỹ tại dân và doanh nghiệp fdi sẽ đi về thứ cho việt để điều tra công ty nhớ đến những người đã được bán đấu.


## 5. Yêu cầu 4 - Đánh giá kết quả sinh câu

**Mục tiêu đánh giá:**
- Chạy tối thiểu 10 lần cho mỗi chế độ (A_word, B_sentence).
- Đo số từ thực tế mỗi câu.
- Tính tỷ lệ đạt ngưỡng độ dài 38-42 từ.
- In ví dụ đầu ra để quan sát chất lượng ngữ nghĩa.

In [23]:
def evaluate_generation(
    bigram_graph,
    reverse_bigram_graph,
    trigram_graph,
    start_counter,
    token_freq,
    tokenized_data,
    n_runs=10,
):
    rows = []

    vi_diacritic_pattern = re.compile(r"[àáảãạăắằẳẵặâấầẩẫậđèéẻẽẹêếềểễệìíỉĩịòóỏõọôốồổỗộơớờởỡợùúủũụưứừửữựỳýỷỹỵ]")

    # Chỉ lấy seed từ tiếng Việt có dấu
    candidate_words = [w for w in bigram_graph.keys() if vi_diacritic_pattern.search(w)]
    if not candidate_words:
        raise ValueError("Không tìm thấy từ tiếng Việt có dấu trong bigram_graph để đánh giá.")

    if len(candidate_words) < n_runs:
        seed_words = [random.choice(candidate_words) for _ in range(n_runs)]
    else:
        seed_words = random.sample(candidate_words, n_runs)

    if len(tokenized_data) < n_runs:
        sampled_sent_tokens = [random.choice(tokenized_data) for _ in range(n_runs)]
    else:
        sampled_sent_tokens = random.sample(tokenized_data, n_runs)

    seed_sentences = []
    for toks in sampled_sent_tokens:
        cut = min(len(toks), random.randint(6, 14))
        seed_sentences.append(" ".join(toks[:cut]))

    for w in seed_words:
        r = generate_from_word(
            w,
            bigram_graph,
            reverse_bigram_graph,
            trigram_graph,
            start_counter,
            token_freq,
        )
        rows.append(
            {
                "mode": "A_word",
                "input": w,
                "word_count": r["word_count"],
                "pass_len": MIN_WORDS <= r["word_count"] <= MAX_WORDS,
                "output": r["sentence"],
            }
        )

    for s in seed_sentences:
        r = generate_from_sentence(
            s,
            bigram_graph,
            reverse_bigram_graph,
            trigram_graph,
            start_counter,
            token_freq,
        )
        rows.append(
            {
                "mode": "B_sentence",
                "input": s,
                "word_count": r["word_count"],
                "pass_len": MIN_WORDS <= r["word_count"] <= MAX_WORDS,
                "output": r["sentence"],
            }
        )

    return pd.DataFrame(rows)


results_df = evaluate_generation(
    bigram_graph=bigram_graph,
    reverse_bigram_graph=reverse_bigram_graph,
    trigram_graph=trigram_graph,
    start_counter=start_counter,
    token_freq=vocab_counter,
    tokenized_data=tokenized_sentences,
    n_runs=10,
)

summary_df = (
    results_df.groupby("mode", as_index=False)
    .agg(
        total_cases=("mode", "count"),
        pass_cases=("pass_len", "sum"),
        avg_words=("word_count", "mean"),
    )
)
summary_df["pass_rate"] = (summary_df["pass_cases"] / summary_df["total_cases"] * 100).round(2)

print("=== TÓM TẮT ĐÁNH GIÁ ===")
print(summary_df)
print()

print("=== MỘT SỐ CÂU SINH MINH HỌA (TOP 6) ===")
best_examples = results_df.sort_values(by=["pass_len", "word_count"], ascending=[False, False]).head(6)
for idx, row in best_examples.reset_index(drop=True).iterrows():
    print(f"[{idx + 1}] Mode={row['mode']} | words={row['word_count']} | pass={row['pass_len']}")
    print("Input:", row["input"])
    print("Output:", row["output"])
    print("-")

results_df.head(10)

=== TÓM TẮT ĐÁNH GIÁ ===
         mode  total_cases  pass_cases  avg_words  pass_rate
0      A_word           10          10       40.0      100.0
1  B_sentence           10          10       40.0      100.0

=== MỘT SỐ CÂU SINH MINH HỌA (TOP 6) ===
[1] Mode=A_word | words=40 | pass=True
Input: hấu
Output: Hấu giúp nông dân trồng ớt trắng giòn của bánh xèo miền tây nam cấp cho để em đi khắp thế giới về công quảng ngãi bị tàu lạ đâm chìm trên dưa chuột và thử sức với số lượng.
-
[2] Mode=A_word | words=40 | pass=True
Input: nhùn
Output: Nhùn tỉnh lai châu có một số nhà nậm pồ trên đường phố việt nam và sìn hồ cảm giác mạnh những kết quả thi đấu của các tổ chức hội nghị thượng đỉnh về khí hậu cho hay đã.
-
[3] Mode=A_word | words=40 | pass=True
Input: marốc
Output: Marốc đối phó với bão số đã đi nhấc bổng cá sấu khổng lồ công nghệ tại việt nam có thể gây tử vong và người yêu vì sợ tốn kém hơn kiểu tập liên hợp quốc về luật giao.
-
[4] Mode=A_word | words=40 | pass=True
Input: ghiền
Outpu

,mode,input,word_count,pass_len,output
0,A_word,hấu,40,True,Hấu giúp nông dân trồng ớt trắng giòn của bánh xèo miền tây nam cấp cho để em đi khắp thế giới về công quảng ngãi bị tàu lạ đâm chìm trên dưa chuột và thử sức với số lượng.
1,A_word,nhùn,40,True,Nhùn tỉnh lai châu có một số nhà nậm pồ trên đường phố việt nam và sìn hồ cảm giác mạnh những kết quả thi đấu của các tổ chức hội nghị thượng đỉnh về khí hậu cho hay đã.
2,A_word,marốc,40,True,Marốc đối phó với bão số đã đi nhấc bổng cá sấu khổng lồ công nghệ tại việt nam có thể gây tử vong và người yêu vì sợ tốn kém hơn kiểu tập liên hợp quốc về luật giao.
3,A_word,ghiền,40,True,Ghiền săn sale bất tận không nhiều nhưng mạng lưới của bundesliga với bàn thắng trong luôn tràn đầy năng lượng hạt nhân iran vì đây là một sự kiện này sẽ kardashian đã rạn nứt sau lũ dữ đang.
4,A_word,nảy,40,True,Nảy sinh tình anh em và niềm mơ cãi về chuyện ấy của mình trong một chiến sĩ đã có chủ quyền biển đảo ra sao nếu là người đầu tiên tại luận với nga để bàn giao cho cơ.
5,A_word,nhẫy,40,True,Nhẫy sau dao kéo của mỹ đã có như vậy sẽ khiến các nhà cung cấp bóng để tham dự hội nghị thượng đỉnh trong gần năm nay là công trình giao nhụa khó ăn uống khoa học và làm.
6,A_word,huyên,40,True,Huyên một chương trình đã phát sóng của văn phòng chính phủ và người có thu gây sốc về vụ việc đang làm rõ nguyễn thị bích hằng nói gì trong ngày trò chơi vương quyền bị tuyên bố sẽ.
7,A_word,hđxx,40,True,Hđxx sơ thẩm vụ án khởi tố bắt đã về nước và quốc tế của trung cho các đối tượng trong nhóm này không tand cấp cao tại hà nội chịu trách hội đồng thi đà nẵng đến bình định.
8,A_word,bống,40,True,Bống dừa là thần tượng kpop có thể hồng phúc và thành phố hồ chí minh lê phong linh không thích điều đó sẽ cá tính cho các công ty con của bé gái tuổi người argentina đã tiếp sức.
9,A_word,chấm,40,True,Chấm điểm real lên kế hoạch của thành nước nhập khẩu và được các cơ quan ban ngành trung ương đã có những dấu thi đấu tại vòng chung kết cuộc đua bi kịch khi đang là chủ tịch xã.


## 6. Tóm tắt nhanh sau tối ưu

Cell này in báo cáo rút gọn:
- Tổng số case theo từng chế độ.
- Tỷ lệ đạt độ dài mục tiêu.
- Một số câu mẫu để kiểm tra chất lượng sinh.

In [24]:
compact_summary = (
    results_df.groupby("mode", as_index=False)
    .agg(
        total_cases=("mode", "count"),
        pass_cases=("pass_len", "sum"),
        avg_words=("word_count", "mean"),
    )
)
compact_summary["pass_rate"] = (compact_summary["pass_cases"] / compact_summary["total_cases"] * 100).round(2)

print("=== SUMMARY SAU TỐI ƯU ===")
print(compact_summary.to_string(index=False))

print("\n=== 3 VÍ DỤ CÂU SINH (RÚT GỌN) ===")
show_df = results_df.head(3).copy()
for i, row in show_df.iterrows():
    out = row["output"]
    if len(out) > 220:
        out = out[:220] + "..."
    print(f"[{i+1}] {row['mode']} | words={row['word_count']} | pass={row['pass_len']}")
    print("Input:", row["input"])
    print("Output:", out)
    print("-")

=== SUMMARY SAU TỐI ƯU ===
      mode  total_cases  pass_cases  avg_words  pass_rate
    A_word           10          10       40.0      100.0
B_sentence           10          10       40.0      100.0

=== 3 VÍ DỤ CÂU SINH (RÚT GỌN) ===
[1] A_word | words=40 | pass=True
Input: hấu
Output: Hấu giúp nông dân trồng ớt trắng giòn của bánh xèo miền tây nam cấp cho để em đi khắp thế giới về công quảng ngãi bị tàu lạ đâm chìm trên dưa chuột và thử sức với số lượng.
-
[2] A_word | words=40 | pass=True
Input: nhùn
Output: Nhùn tỉnh lai châu có một số nhà nậm pồ trên đường phố việt nam và sìn hồ cảm giác mạnh những kết quả thi đấu của các tổ chức hội nghị thượng đỉnh về khí hậu cho hay đã.
-
[3] A_word | words=40 | pass=True
Input: marốc
Output: Marốc đối phó với bão số đã đi nhấc bổng cá sấu khổng lồ công nghệ tại việt nam có thể gây tử vong và người yêu vì sợ tốn kém hơn kiểu tập liên hợp quốc về luật giao.
-


In [ ]:
---
## Kết luận

### Kết quả chính
- Hoàn thành pipeline sinh câu tiếng Việt có dấu từ dữ liệu thực tế.
- Hỗ trợ đầy đủ 2 chế độ đầu vào: 1 từ và 1 câu.
- Đảm bảo chống lặp vô hạn và có fallback khi dead-end.
- Duy trì độ dài mục tiêu ~40 từ với tỷ lệ đạt cao.

### Các cải tiến đã áp dụng
- Lọc nhiễu token theo tần suất và đặc trưng tiếng Việt.
- Kết hợp Bigram + Trigram để tăng mượt ngữ cảnh.
- Bổ sung topic coherence để giảm nhảy chủ đề.
- Giới hạn seed từ đơn theo tiếng Việt có dấu.

### Hướng phát triển tiếp theo
- Bổ sung bước hậu xử lý ngữ pháp (grammar polish).
- Thử Beam Search để tăng chất lượng câu.
- So sánh với mô hình neural (LSTM/Transformer) cho cùng bài toán.